# HW3 — Two retrievers and a model: what RAG actually sends

**Name:**
**Student ID:**
**Group:**
**Repository:**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 |large, language, models, can, generate, text, and, answer, questions|9|
| D2 |retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer|10|
| D3 |embeddings, represent, text, as, numerical, vectors|6|

**avgdl =** 8.333

**Query terms:**

**TF, DF and IDF of each query term:**

| Query term | f(t,D1) | f(t,D2) | f(t,D3) | n(t) | idf(t) |
| --- | --- | --- | --- | --- | --- |
| how | 0 | 0 | 0 | 0 | 1.946 |
| does | 0 | 0 | 0 | 0 | 1.946 |
| rag | 0 | 0 | 0 | 0 | 1.946 |
| retrieve | 0 | 0 | 0 | 0 | 1.946 |
| information | 0 | 0 | 0 | 0 | 1.946 |

**BM25 of each document** (write out the sum for every document, one line per term that contributes):
/
>

| Document | BM25 Score | Rank |
| --- | --- | --- |
| D1 | 0.000 | 2.0 |
| D2 | 0.000 | 2.0 |
| D3 | 0.000 | 2.0 |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|-----|--------|----|
| D1  | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2  | retrieve, augmented, gener, retrieve, relevant, documents, before, gener, an, answer | 10 |
| D3  | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** 8.333

**Stemmed query terms:**

| Query term | f(t,D1) | f(t,D2) | f(t,D3) | n(t) | idf(t) |
| --- | --- | --- | --- | --- | --- |
| how | 0 | 0 | 0 | 0 | 1.946 |
| doe | 0 | 0 | 0 | 0 | 1.946 |
| rag | 0 | 0 | 0 | 0 | 1.946 |
| retriev | 0 | 2 | 0 | 1 | 0.511 |
| inform | 0 | 0 | 0 | 0 | 1.946 |

**BM25 of each document**, worked:

>

| Document | BM25 Score | Rank |
| --- | --- | --- |
| D1 | 0.000 | 2.5 |
| D2 | 0.686 | 1.0 |
| D3 | 0.000 | 2.5 |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> In (a), BM25 does not single out any document (all are equal).
In (b), BM25 retrieves D2 first, as it is the only one containing the query term (retriev).

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

| Pair | Dot product | ‖Query‖ | ‖D‖ | Cosine |
|------|-------------|-----------|-------|--------|
| 0 Query‑D1 | 0.78 | 0.964 | 0.812 | 0.996 |
| 1 Query‑D2 | 1.08 | 0.964 | 1.122 | 0.998 |
| 2 Query‑D3 | 0.34 | 0.964 | 0.831 | 0.424 |

| Document | Cosine Similarity | Rank |
| --- | --- | --- |
| D2 | 0.998 | 1 |
| D1 | 0.996 | 2 |
| D3 | 0.424 | 3 |

**Which document is most similar to the query?**

> Document D2 is the most similar to the query (cosine similarity = 0.998).

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Cosine: D2 → D1 → D3.
BM25 (A1 a): D1 = D2 = D3 (tie).
BM25 (A1 b, with stemming): D2 → D1 = D3.

The ranking does not match BM25 completely—only the first-place ranking of D2 is the same. The difference lies in positions 2 and 3: BM25 ranks D1 and D3 as a tie, whereas Cosine distinguishes between them.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

BM25 looks at exact terms: the query contains "retriev" (retrieve), which appears only in D2, so BM25 assigns a high score to D2.

Embeddings (cosine similarity) look at meaning: the words in D1—generate, language, models, answer, text—are semantically closer to the query, so D1 ranks higher than D3.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
#A1 (a)
import math
import pandas as pd

docs = {
    "D1": "large language models can generate text and answer questions",
    "D2": "retrieval-augmented generation retrieves relevant documents before generating an answer",
    "D3": "embeddings represent text as numerical vectors"
}
query = "how does rag retrieve information"

def tokenize(text):
    return text.lower().replace("-", " ").split()

tokens_query = tokenize(query)
tokens_docs = {d: tokenize(text) for d, text in docs.items()}

# Таблица токенов и длин
lengths = {d: len(tokens_docs[d]) for d in docs}
df_lengths = pd.DataFrame({
    "Doc": list(lengths.keys()),
    "Tokens": [", ".join(tokens_docs[d]) for d in docs],  # токены через запятую
    "|d|": list(lengths.values())
})
display(df_lengths)

avgdl = sum(lengths.values()) / len(lengths)
print("avgdl =", round(avgdl,3))

# TF, DF, IDF
tf = {d: {t: tokens_docs[d].count(t) for t in tokens_query} for d in docs}
df = {t: sum(1 for d in docs if t in tokens_docs[d]) for t in tokens_query}
N = len(docs)
idf = {t: math.log((N - df[t] + 0.5) / (df[t] + 0.5)) for t in tokens_query}

df_tf = pd.DataFrame({
    "Query term": tokens_query,
    "f(t,D1)": [tf["D1"][t] for t in tokens_query],
    "f(t,D2)": [tf["D2"][t] for t in tokens_query],
    "f(t,D3)": [tf["D3"][t] for t in tokens_query],
    "n(t)": [df[t] for t in tokens_query],
    "idf(t)": [round(idf[t],3) for t in tokens_query]
})
display(df_tf)

# BM25
k1, b = 1.5, 0.75
scores = {}
for d in docs:
    score = 0
    for t in tokens_query:
        f = tf[d][t]
        if f == 0: continue
        denom = f + k1 * (1 - b + b * lengths[d] / avgdl)
        score += idf[t] * f * (k1 + 1) / denom
    scores[d] = score

df_scores = pd.DataFrame({
    "Document": list(scores.keys()),
    "BM25 Score": [round(scores[d],3) for d in scores],
})
df_scores["Rank"] = df_scores["BM25 Score"].rank(ascending=False)
display(df_scores)

,Doc,Tokens,|d|
0,D1,"large, language, models, can, generate, text, ...",9
1,D2,"retrieval, augmented, generation, retrieves, r...",10
2,D3,"embeddings, represent, text, as, numerical, ve...",6


avgdl = 8.333


,Query term,"f(t,D1)","f(t,D2)","f(t,D3)",n(t),idf(t)
0,how,0,0,0,0,1.946
1,does,0,0,0,0,1.946
2,rag,0,0,0,0,1.946
3,retrieve,0,0,0,0,1.946
4,information,0,0,0,0,1.946


,Document,BM25 Score,Rank
0,D1,0,2.0
1,D2,0,2.0
2,D3,0,2.0


In [2]:
#A1 (b)
import math
import pandas as pd

docs = {
    "D1": "large language models can generate text and answer questions",
    "D2": "retrieval-augmented generation retrieves relevant documents before generating an answer",
    "D3": "embeddings represent text as numerical vectors"
}
query = "how does rag retrieve information"

def tokenize(text):
    return text.lower().replace("-", " ").split()

# словарь стемминга
stems = {
    "retrieve":"retriev","retrieves":"retriev","retrieval":"retriev",
    "generate":"gener","generating":"gener","generation":"gener",
    "information":"inform","does":"doe"
}

def stem_tokens(tokens):
    return [stems.get(t, t) for t in tokens]

tokens_query = tokenize(query)
tokens_query_stem = stem_tokens(tokens_query)
tokens_docs = {d: tokenize(text) for d, text in docs.items()}
tokens_docs_stem = {d: stem_tokens(tokens_docs[d]) for d in docs}

# Таблица токенов и длин
lengths_stem = {d: len(tokens_docs_stem[d]) for d in docs}
df_lengths = pd.DataFrame({
    "Doc": list(lengths_stem.keys()),
    "Tokens": [", ".join(tokens_docs_stem[d]) for d in docs],  # токены через запятую
    "|d|": list(lengths_stem.values())
})
display(df_lengths)

avgdl_stem = sum(lengths_stem.values()) / len(lengths_stem)
print("avgdl =", round(avgdl_stem,3))

# TF, DF, IDF
tf_stem = {d: {t: tokens_docs_stem[d].count(t) for t in tokens_query_stem} for d in docs}
df_stem = {t: sum(1 for d in docs if t in tokens_docs_stem[d]) for t in tokens_query_stem}
N = len(docs)
idf_stem = {t: math.log((N - df_stem[t] + 0.5) / (df_stem[t] + 0.5)) for t in tokens_query_stem}

df_tf = pd.DataFrame({
    "Query term": tokens_query_stem,
    "f(t,D1)": [tf_stem["D1"][t] for t in tokens_query_stem],
    "f(t,D2)": [tf_stem["D2"][t] for t in tokens_query_stem],
    "f(t,D3)": [tf_stem["D3"][t] for t in tokens_query_stem],
    "n(t)": [df_stem[t] for t in tokens_query_stem],
    "idf(t)": [round(idf_stem[t],3) for t in tokens_query_stem]
})
display(df_tf)

# BM25
k1, b = 1.5, 0.75
scores_stem = {}
for d in docs:
    score = 0
    for t in tokens_query_stem:
        f = tf_stem[d][t]
        if f == 0: continue
        denom = f + k1 * (1 - b + b * lengths_stem[d] / avgdl_stem)
        score += idf_stem[t] * f * (k1 + 1) / denom
    scores_stem[d] = score

df_scores = pd.DataFrame({
    "Document": list(scores_stem.keys()),
    "BM25 Score": [round(scores_stem[d],3) for d in scores_stem],
})
df_scores["Rank"] = df_scores["BM25 Score"].rank(ascending=False)
display(df_scores)

,Doc,Tokens,|d|
0,D1,"large, language, models, can, gener, text, and...",9
1,D2,"retriev, augmented, gener, retriev, relevant, ...",10
2,D3,"embeddings, represent, text, as, numerical, ve...",6


avgdl = 8.333


,Query term,"f(t,D1)","f(t,D2)","f(t,D3)",n(t),idf(t)
0,how,0,0,0,0,1.946
1,doe,0,0,0,0,1.946
2,rag,0,0,0,0,1.946
3,retriev,0,2,0,1,0.511
4,inform,0,0,0,0,1.946


,Document,BM25 Score,Rank
0,D1,0.000,2.5
1,D2,0.686,1.0
2,D3,0.000,2.5


In [3]:
#A2
import numpy as np
import pandas as pd

query = np.array([0.8, 0.2, 0.5])
D1 = np.array([0.7, 0.1, 0.4])
D2 = np.array([0.9, 0.3, 0.6])
D3 = np.array([0.1, 0.8, 0.2])

docs = {"D1": D1, "D2": D2, "D3": D3}

def cosine_similarity(q, d):
    dot = np.dot(q, d)
    norm_q = np.linalg.norm(q)
    norm_d = np.linalg.norm(d)
    cos = dot / (norm_q * norm_d)
    return dot, norm_q, norm_d, cos

# Таблица 1: промежуточные значения
rows = []
for name, vec in docs.items():
    dot, norm_q, norm_d, cos = cosine_similarity(query, vec)
    rows.append([f"Query-{name}", round(dot,3), round(norm_q,3), round(norm_d,3), round(cos,3)])

table1 = pd.DataFrame(rows, columns=["Pair","Dot product","||Query||","||D||","Cosine"])
display(table1)

# Таблица 2: ранжирование
ranking = sorted([(name, cosine_similarity(query, vec)[3]) for name, vec in docs.items()],
                 key=lambda x: x[1], reverse=True)

rows2 = [[doc, round(cos,3), rank+1] for rank,(doc,cos) in enumerate(ranking)]
table2 = pd.DataFrame(rows2, columns=["Document","Cosine Similarity","Rank"])
display(table2)

,Pair,Dot product,||Query||,||D||,Cosine
0,Query-D1,0.78,0.964,0.812,0.996
1,Query-D2,1.08,0.964,1.122,0.998
2,Query-D3,0.34,0.964,0.831,0.424


,Document,Cosine Similarity,Rank
0,D2,0.998,1
1,D1,0.996,2
2,D3,0.424,3


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [4]:
import json
import pandas as pd
from pathlib import Path

corpus_path = Path("data/corpus.jsonl")
queries_path = Path("data/queries.json")

corpus = []
ids = []
with corpus_path.open("r", encoding="utf8") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        ids.append(obj.get("id", str(i)))
        corpus.append(obj.get("text", ""))

with queries_path.open("r", encoding="utf8") as f:
    queries = json.load(f)

# Приводим queries к списку
if isinstance(queries, dict):
    queries_list = list(queries.values())
else:
    queries_list = list(queries)

df_corpus = pd.DataFrame({
    "id": ids,
    "text": corpus
})

df_queries = pd.DataFrame({
    "query": queries_list
})

print("=== Corpus ===")
display(df_corpus.head())

print("=== Queries ===")
display(df_queries.head())

=== Corpus ===


,id,text
0,KZ-01,"Kazakhstan, officially the Republic of Kazakhs..."
1,KZ-02,Kazakhstan is the world's ninth-largest countr...
2,KZ-03,Kazakhstan has been inhabited since the Paleol...
3,KZ-04,Kazakhstan dominates Central Asia both economi...
4,KZ-05,Astana is the capital city of Kazakhstan. With...


=== Queries ===


,query
0,Part B's eight fixed queries. Do not edit them...
1,"[{'id': 'Q-01', 'text': 'What is form GR-305 u..."


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

> My tokenisation: lowercase + regex \w+, stop‑words removed, with stemming.

lowercase — so that "Form" and "form" match.

regex \w+ — simple word extraction.

stop-words — removing unnecessary function words.

stemming — treating different forms of a word as the same.

In [5]:
import json, re
import pandas as pd
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# --- загрузка ресурсов NLTK ---
nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

# --- Настройка pandas: показывать весь текст
pd.set_option("display.max_colwidth", None)

# --- Загрузка корпуса ---
with open("data/corpus.jsonl", "r", encoding="utf8") as f:
    corpus = [json.loads(line) for line in f]

texts = [c["text"] for c in corpus]
ids = [c.get("id", str(i)) for i, c in enumerate(corpus)]

# --- Загрузка фиксированных запросов ---
with open("data/queries.json", "r", encoding="utf8") as f:
    queries_data = json.load(f)

if isinstance(queries_data, dict) and "queries" in queries_data:
    queries = queries_data["queries"]
else:
    queries = queries_data

# --- Токенизация: lowercase + regex + стоп-слова + стемминг ---
def tokenizer(s):
    tokens = re.findall(r"\w+", s.lower())
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

tokenized_corpus = [tokenizer(t) for t in texts]

# --- BM25 индекс ---
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

# --- Функция для топ-3 ---
def bm25_top3(query_text):
    qtok = tokenizer(query_text)
    scores = bm25.get_scores(qtok)
    idx = scores.argsort()[::-1][:3]
    rows = []
    for rank, j in enumerate(idx, start=1):

        first_line = texts[j].splitlines()[0]
        rows.append({
            "rank": rank,
            "chunk_id": ids[j],
            "first_line": first_line,
            "score": float(scores[j])
        })
    return pd.DataFrame(rows)

for q in queries:
    print("\n=== Query", q["id"], ":", q["text"])
    df_scores = bm25_top3(q["text"])
    display(df_scores)


=== Query Q-01 : What is form GR-305 used for?


,rank,chunk_id,first_line,score
0,1,GO-08,Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.,13.402425
1,2,GO-12,"A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application.",8.833345
2,3,GO-06,"Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.",5.868962



=== Query Q-02 : My family is poor. Is there financial help for students like me?


,rank,chunk_id,first_line,score
0,1,KZ-25,"Beshbarmak is usually made from finely chopped boiled meat, mixed with dough (typically egg noodles) and chyk, an onion sauce. It is typically served on large communal platters, shared between several people, after shorpo, a first course of mutton broth served in bowls called kese. It is also followed by a broth called ak-serke (shorpo mixed with kymyz or ayran), which is thought to help with settling the stomach. Festive beshbarmak can be cooked together with qazı and chuchuk.",4.059947
1,2,KZ-13,"Like other Turkic languages, Kazakh is an agglutinative language and employs vowel harmony. Kazakh builds words by adding suffixes one after another to the word stem, with each suffix expressing only one unique meaning and following a fixed sequence. Ethnologue recognises three mutually intelligible dialect groups: North-Eastern Kazakh—the most widely spoken variety, which also serves as the basis for the official language—Southern Kazakh, and Western Kazakh. The language shares a degree of mutual intelligibility with the closely related Karakalpak language, while its Western dialects maintain limited mutual intelligibility with the Altai languages.",3.174489
2,3,KZ-17,"The lake currently covers about 16,400 km2 (6,300 sq mi). However, like the Aral Sea, it is shrinking due to diversion and extraction of water from its feeders. The lake has a narrow, quite central, strait. The lake's western part is fresh water and its eastern half is saline. The eastern part is on average 1.7 times deeper than the west. The largest shore city is named Balkhash and has about 66,000 inhabitants. Main local economic activities include mining, ore processing and fishing. There is concern about the lake's shallowing due to desertification of microclimates and water extraction for multiplied industrial output. Moreover, the impacts of climate change may also negatively affect the lake and its ecosystems.",2.876209



=== Query Q-03 : Өтінішті қашанға дейін тапсыру керек?


...

,rank,chunk_id,first_line,score
0,1,KZ-37,"The Semipalatinsk Test Site or Semipalatinsk-21 (Russian: Семипалатинск-21; Kazakh: Семей-21, romanized: Semei-21), also known as ""The Polygon"", was the primary testing venue for the Soviet Union's nuclear weapons. It is located in Zhanasemey District, Abai Region, Kazakhstan, south of the valley of the Irtysh River. The test site was part of the former Kazakh SSR. The scientific buildings for the test site were located around 150 km (93 mi) west of the town of Semipalatinsk, later renamed Semey, near the border of East Kazakhstan Region and Pavlodar Region. Most of the nuclear tests took place at various sites further to the west and the south, some as far as into Karagandy Region.",17.788969
1,2,KZ-38,"The Soviet Union conducted 456 nuclear tests at Semipalatinsk from 1949 until 1989 with little regard for their effect on the local people or environment. The full impact of radiation exposure was hidden for many years by Soviet authorities and has only come to light since the test site closed in 1991. According to estimates from Kazakh experts, 1.5 million people were exposed to fallout over the years. From 1996 to 2012, a secret joint operation of Kazakh, Russian, and American nuclear scientists and engineers secured some waste plutonium in the tunnels of the mountains. The Semipalatinsk Test Site saw the detonation of the first Soviet atom bomb and the first air-tested hydrogen bomb. Over the course of 40 years, a quarter of all nuclear tests in history took place here. Since its closure on 29 August 1991, the Semipalatinsk Test Site has become the best-researched nuclear testing site in the world, and the only one in the world open to the public year-round.",15.997001
2,3,KZ-03,"Kazakhstan has been inhabited since the Paleolithic era. In antiquity, various nomadic Iranian peoples such as the Saka, Massagetae, and Scythians. Turkic nomads entered the region from the sixth century. In the 13th century, the area was subjugated by the Mongol Empire under Genghis Khan. Following the disintegration of the Golden Horde in the 15th century, the Kazakh Khanate was established over an area roughly corresponding with modern Kazakhstan. By the 18th century, the Kazakh Khanate had fragmented into three jüz (tribal divisions), which were gradually absorbed and conquered by the Russian Empire; by the mid-19th century, all of Kazakhstan was nominally under Russian rule. Following the 1917 Russian Revolution and subsequent Russian Civil War, it became an autonomous republic of the Russian SFSR within the Soviet Union. Its status was elevated to that of a union republic in 1936. The Soviet government settled Russians and other ethnicities in the republic, which resulted in ethnic Kazakhs being a minority during the Soviet era. Kazakhstan was the last constituent republic of the Soviet Union to declare independence in 1991 during its dissolution.",6.247462



=== Query Q-07 : Can the grant be sent to my mother's card?


,rank,chunk_id,first_line,score
0,1,GO-21,"The Grant Office is in room 214 of the main building and is open Monday to Friday from 10:00 to 17:00, with a lunch break from 13:00 to 14:00. Questions can also be sent through the 'Grants' section of the student portal, which is answered within three working days. The front desk can explain the rules but cannot change a decision or accept documents on paper.",6.217535
1,2,GO-02,"An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.",5.692682
2,3,GO-06,"Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.",5.333202



=== Query Q-08 : If my grades drop in the autumn, do I still get paid in February?


,rank,chunk_id,first_line,score
0,1,GO-17,"The grant continues only while the student keeps up their studies. If the cumulative GPA falls below 2.67 at the end of the autumn semester, the February instalment is suspended. It is not reinstated later in the same year, even if the GPA recovers. The October instalment does not have to be returned, unless the student withdraws from the university before 1 December, in which case it must be repaid in full.",8.052437
1,2,GO-04,"The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.",6.554974
2,3,GO-01,"The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan: it does not have to be paid back unless the student withdraws early, as explained in the section on academic progress.",4.573533


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [8]:
import os
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from IPython.display import display

# --- Отключаем предупреждение про симлинки ---
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

with open("data/corpus.jsonl", "r", encoding="utf8") as f:
    corpus = [json.loads(line) for line in f]

texts = [c["text"] for c in corpus]
ids = [c.get("id", str(i)) for i, c in enumerate(corpus)]

with open("data/queries.json", "r", encoding="utf8") as f:
    queries_data = json.load(f)
queries = queries_data["queries"] if isinstance(queries_data, dict) else queries_data

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

doc_embeddings = model.encode(texts, convert_to_tensor=True)

# --- Функция для топ-3 ---
def embed_top3(query_text):
    query_emb = model.encode(query_text, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, doc_embeddings)[0].cpu().numpy()
    idx = scores.argsort()[::-1][:3]
    rows = []
    for rank, j in enumerate(idx, start=1):
        first_line = texts[j].splitlines()[0]
        rows.append({
            "rank": rank,
            "chunk_id": ids[j],
            "first_line": first_line,
            "score": float(scores[j])
        })
    return pd.DataFrame(rows)

for q in queries:
    print("\n=== Query", q["id"], ":", q["text"])
    df_scores = embed_top3(q["text"])
    display(df_scores)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2244.71it/s]



=== Query Q-01 : What is form GR-305 used for?


,rank,chunk_id,first_line,score
0,1,GO-08,Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.,0.694203
1,2,GO-06,"Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.",0.565137
2,3,GO-07,"Form GR-210 is the household income declaration. It lists every adult living in the household and the monthly income of each, and it must be signed by all of them. The Grant Office divides the total household income by the number of people in the household to find the income per person, which sets the band. A GR-210 with a missing signature is returned unprocessed.",0.539778



=== Query Q-02 : My family is poor. Is there financial help for students like me?


,rank,chunk_id,first_line,score
0,1,GO-01,"The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan: it does not have to be paid back unless the student withdraws early, as explained in the section on academic progress.",0.504109
1,2,GO-03,"The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their students may still apply for merit scholarships, which are run separately.",0.365993
2,3,GO-16,"The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.",0.345146


...

,rank,chunk_id,first_line,score
0,1,KZ-38,"The Soviet Union conducted 456 nuclear tests at Semipalatinsk from 1949 until 1989 with little regard for their effect on the local people or environment. The full impact of radiation exposure was hidden for many years by Soviet authorities and has only come to light since the test site closed in 1991. According to estimates from Kazakh experts, 1.5 million people were exposed to fallout over the years. From 1996 to 2012, a secret joint operation of Kazakh, Russian, and American nuclear scientists and engineers secured some waste plutonium in the tunnels of the mountains. The Semipalatinsk Test Site saw the detonation of the first Soviet atom bomb and the first air-tested hydrogen bomb. Over the course of 40 years, a quarter of all nuclear tests in history took place here. Since its closure on 29 August 1991, the Semipalatinsk Test Site has become the best-researched nuclear testing site in the world, and the only one in the world open to the public year-round.",0.661244
1,2,KZ-37,"The Semipalatinsk Test Site or Semipalatinsk-21 (Russian: Семипалатинск-21; Kazakh: Семей-21, romanized: Semei-21), also known as ""The Polygon"", was the primary testing venue for the Soviet Union's nuclear weapons. It is located in Zhanasemey District, Abai Region, Kazakhstan, south of the valley of the Irtysh River. The test site was part of the former Kazakh SSR. The scientific buildings for the test site were located around 150 km (93 mi) west of the town of Semipalatinsk, later renamed Semey, near the border of East Kazakhstan Region and Pavlodar Region. Most of the nuclear tests took place at various sites further to the west and the south, some as far as into Karagandy Region.",0.588830
2,3,KZ-10,"The Baikonur Cosmodrome is a spaceport operated by Russia within Kazakhstan. Located in the Kazakh city of Baikonur, it is the largest operational space launch facility in terms of area. All Russian crewed spaceflights are launched from Baikonur. Situated in the Kazakh Steppe, some 90 metres (300 ft) above sea level, it is 200 kilometres (120 mi) to the east of the Aral Sea and 33 kilometres (21 mi) north of the Syr Darya. It is close to Töretam, a station on the Trans-Aral Railway. Russia, as the official successor state to the Soviet Union, has retained control over the facility since 1991; it originally assumed this role through the post-Soviet Commonwealth of Independent States (CIS), but ratified an agreement with Kazakhstan in 2005 that allowed it to lease the spaceport until 2050. It is jointly managed by Roscosmos and the Russian Aerospace Forces.",0.368953



=== Query Q-07 : Can the grant be sent to my mother's card?


,rank,chunk_id,first_line,score
0,1,GO-02,"An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.",0.439904
1,2,GO-21,"The Grant Office is in room 214 of the main building and is open Monday to Friday from 10:00 to 17:00, with a lunch break from 13:00 to 14:00. Questions can also be sent through the 'Grants' section of the student portal, which is answered within three working days. The front desk can explain the rules but cannot change a decision or accept documents on paper.",0.410225
2,3,GO-20,"The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules.",0.409668



=== Query Q-08 : If my grades drop in the autumn, do I still get paid in February?


,rank,chunk_id,first_line,score
0,1,GO-17,"The grant continues only while the student keeps up their studies. If the cumulative GPA falls below 2.67 at the end of the autumn semester, the February instalment is suspended. It is not reinstated later in the same year, even if the GPA recovers. The October instalment does not have to be returned, unless the student withdraws from the university before 1 December, in which case it must be repaid in full.",0.535928
1,2,GO-18,"A student who goes on academic leave keeps their grant decision, but payments stop for the months of the leave and resume in the next instalment after they return. A student who transfers to a different programme inside the university keeps the grant. A student who transfers to another university loses it; the grant cannot be moved with them.",0.317057
2,3,GO-19,"The grant budget is fixed each year. If there are more qualifying applicants than funds, grants are awarded in this order: band 1 applicants before band 2 applicants; within a band, the higher GPA first; and when two applicants have the same band and the same GPA, the earlier GR-104 submission time wins. Qualifying applicants who miss out are placed on a waiting list for the February round.",0.310518


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
| --- | --- | --- | --- |
| Q-09 | GR‑104 form submission | GO‑19 | **BM25** — точное совпадение по фразе «GR‑104 submission», ключевые слова работают лучше |
| Q-10 | How to declare family income | GO‑07 | **Embeddings** — понимают смысл и перефраз («family income» → «household income declaration») |

In [12]:
import os
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
from IPython.display import display

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

with open("data/corpus.jsonl", "r", encoding="utf8") as f:
    corpus = [json.loads(line) for line in f]

texts = [c["text"] for c in corpus]
ids = [c.get("id", str(i)) for i, c in enumerate(corpus)]

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_embeddings = model.encode(texts, convert_to_tensor=True)

tokenized_corpus = [t.split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_best(query_text):
    tokenized_query = query_text.split()
    scores = bm25.get_scores(tokenized_query)
    idx = scores.argsort()[::-1][:3]  # топ-3
    rows = []
    for rank, j in enumerate(idx, start=1):
        rows.append({"rank": rank, "chunk_id": ids[j], "first_line": texts[j].splitlines()[0], "score": float(scores[j])})
    return pd.DataFrame(rows)

def embed_best(query_text):
    query_emb = model.encode(query_text, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, doc_embeddings)[0].cpu().numpy()
    idx = scores.argsort()[::-1][:3]  # топ-3
    rows = []
    for rank, j in enumerate(idx, start=1):
        rows.append({"rank": rank, "chunk_id": ids[j], "first_line": texts[j].splitlines()[0], "score": float(scores[j])})
    return pd.DataFrame(rows)

queries = [
    {"id": "Q-09", "text": "GR-104 form submission", "method": "bm25"},
    {"id": "Q-10", "text": "How to declare family income", "method": "embeddings"}
]

for q in queries:
    print(f"\n=== {q['id']} : {q['text']}")
    if q["method"] == "bm25":
        print("BM25 best results:")
        display(bm25_best(q["text"]))
    else:
        print("Embeddings best results:")
        display(embed_best(q["text"]))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3258.19it/s]



=== Q-09 : GR-104 form submission
BM25 best results:


,rank,chunk_id,first_line,score
0,1,GO-19,"The grant budget is fixed each year. If there are more qualifying applicants than funds, grants are awarded in this order: band 1 applicants before band 2 applicants; within a band, the higher GPA first; and when two applicants have the same band and the same GPA, the earlier GR-104 submission time wins. Qualifying applicants who miss out are placed on a waiting list for the February round.",7.497179
1,2,GO-06,"Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.",6.758454
2,3,GO-14,"2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін студенттік портал арқылы қабылданады. Өтініш беру үшін GR-104 нысанын толтырып, транскрипт пен жеке куәліктің көшірмесін тіркеу керек. Мерзімі өткен өтініштер қаралмайды. Сұрақтарыңыз болса, Грант бөліміне хабарласыңыз.",3.779879



=== Q-10 : How to declare family income
Embeddings best results:


,rank,chunk_id,first_line,score
0,1,GO-07,"Form GR-210 is the household income declaration. It lists every adult living in the household and the monthly income of each, and it must be signed by all of them. The Grant Office divides the total household income by the number of people in the household to find the income per person, which sets the band. A GR-210 with a missing signature is returned unprocessed.",0.563824
1,2,GO-10,"Proof of income must accompany form GR-210. Accepted: a salary certificate from the employer issued within the last three months, a pension statement, a certificate of registration as unemployed, or an official statement of social benefits. Not accepted: screenshots of a bank app, handwritten notes, messages from an employer, or self-declared income without a document. An income that is not proven is not counted in the student's favour.",0.538801
2,3,GO-03,"The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their students may still apply for merit scholarships, which are run separately.",0.412969


## Results for all ten or more queries: top 3 from each retriever

In [10]:
import os
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
from IPython.display import display

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

with open("data/corpus.jsonl", "r", encoding="utf8") as f:
    corpus = [json.loads(line) for line in f]

texts = [c["text"] for c in corpus]
ids = [c.get("id", str(i)) for i, c in enumerate(corpus)]

with open("data/queries.json", "r", encoding="utf8") as f:
    queries_data = json.load(f)
queries = queries_data["queries"] if isinstance(queries_data, dict) else queries_data

queries.extend([
    {"id": "Q-09", "text": "GR-104 form submission", "relevant": "GO-06"},
    {"id": "Q-10", "text": "How to declare family income", "relevant": "GO-07"}
])

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_embeddings = model.encode(texts, convert_to_tensor=True)

tokenized_corpus = [t.split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)

def embed_top3(query_text):
    query_emb = model.encode(query_text, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, doc_embeddings)[0].cpu().numpy()
    idx = scores.argsort()[::-1][:3]
    rows = []
    for rank, j in enumerate(idx, start=1):
        first_line = texts[j].splitlines()[0]
        rows.append({
            "rank": rank,
            "chunk_id": ids[j],
            "first_line": first_line,
            "score": float(scores[j])
        })
    return pd.DataFrame(rows)

def bm25_top3(query_text):
    tokenized_query = query_text.split()
    scores = bm25.get_scores(tokenized_query)
    idx = scores.argsort()[::-1][:3]
    rows = []
    for rank, j in enumerate(idx, start=1):
        first_line = texts[j].splitlines()[0]
        rows.append({
            "rank": rank,
            "chunk_id": ids[j],
            "first_line": first_line,
            "score": float(scores[j])
        })
    return pd.DataFrame(rows)

for q in queries:
    print("\n=== Query", q["id"], ":", q["text"])
    print("BM25 top-3:")
    display(bm25_top3(q["text"]))
    print("Embeddings top-3:")
    display(embed_top3(q["text"]))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2046.07it/s]



=== Query Q-01 : What is form GR-305 used for?
BM25 top-3:


,rank,chunk_id,first_line,score
0,1,GO-08,Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.,12.963887
1,2,KZ-43,"The Kazakh language has been written mainly in four scripts at various points of time—Old Turkic, Cyrillic, Latin, and Arabic—each having a distinct alphabet. The Arabic script is used in Iran, Afghanistan, and China, while the Cyrillic script is used in Kazakhstan, Kyrgyzstan, Russia, and Mongolia. In October 2017, a presidential decree in Kazakhstan ordered a transition from the Cyrillic to Latin script to be implemented by 2025. In January 2021, the target year for finishing the transition was pushed back to 2031.",4.368658
2,3,GO-06,"Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.",4.318055


Embeddings top-3:


,rank,chunk_id,first_line,score
0,1,GO-08,Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.,0.694203
1,2,GO-06,"Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.",0.565137
2,3,GO-07,"Form GR-210 is the household income declaration. It lists every adult living in the household and the monthly income of each, and it must be signed by all of them. The Grant Office divides the total household income by the number of people in the household to find the income per person, which sets the band. A GR-210 with a missing signature is returned unprocessed.",0.539778


...

,rank,chunk_id,first_line,score
0,1,GO-06,"Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.",0.703883
1,2,GO-08,Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.,0.549091
2,3,GO-07,"Form GR-210 is the household income declaration. It lists every adult living in the household and the monthly income of each, and it must be signed by all of them. The Grant Office divides the total household income by the number of people in the household to find the income per person, which sets the band. A GR-210 with a missing signature is returned unprocessed.",0.454370



=== Query Q-10 : How to declare family income
BM25 top-3:


,rank,chunk_id,first_line,score
0,1,GO-07,"Form GR-210 is the household income declaration. It lists every adult living in the household and the monthly income of each, and it must be signed by all of them. The Grant Office divides the total household income by the number of people in the household to find the income per person, which sets the band. A GR-210 with a missing signature is returned unprocessed.",5.021829
1,2,GO-03,"The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their students may still apply for merit scholarships, which are run separately.",4.194808
2,3,GO-22,"The documents an applicant uploads are used only to decide the grant. They are kept for three years after the last payment and then deleted. The Grant Office does not share income information with lecturers, with other students, or with any organisation outside the university, except where the law requires it.",3.704404


Embeddings top-3:


,rank,chunk_id,first_line,score
0,1,GO-07,"Form GR-210 is the household income declaration. It lists every adult living in the household and the monthly income of each, and it must be signed by all of them. The Grant Office divides the total household income by the number of people in the household to find the income per person, which sets the band. A GR-210 with a missing signature is returned unprocessed.",0.563824
1,2,GO-10,"Proof of income must accompany form GR-210. Accepted: a salary certificate from the employer issued within the last three months, a pension statement, a certificate of registration as unemployed, or an official statement of social benefits. Not accepted: screenshots of a bank app, handwritten notes, messages from an employer, or self-declared income without a document. An income that is not proven is not counted in the student's favour.",0.538801
2,3,GO-03,"The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their students may still apply for merit scholarships, which are run separately.",0.412969


## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO‑08, KZ‑43, GO‑06 | GO‑08 | GO‑08, GO‑06, GO‑07 | GO‑08 |
| Q-02 | GO‑20, KZ‑25, KZ‑17 | - | GO‑01, GO‑03, GO‑16 | GO‑01 |
| Q-03 | GO‑14, GO‑23, GO‑22 | GO‑14 | GO‑14, KZ‑12, KZ‑10 | GO‑14 |
| Q-04 | GO‑05, GO‑17, GO‑02 | GO‑02 | GO‑05, GO‑02, GO‑04 | GO‑02 |
| Q-05 | KZ‑05, KZ‑11, KZ‑33 | KZ‑05 | KZ‑05, KZ‑01, KZ‑03 | KZ‑05 |
| Q-06 | KZ‑38, KZ‑37, KZ‑03 | KZ‑37, KZ‑38 | KZ‑38, KZ‑37, KZ‑10 | KZ‑37, KZ‑38 |
| Q-07 | GO‑21, GO‑18, GO‑16 | - | GO‑02, GO‑21, GO‑20 | - |
| Q-08 | GO‑05, GO‑04, GO‑11 | - | GO‑17, GO‑18, GO‑19 | GO‑17 |
| Q-09 | GO‑19, GO‑06, GO‑14 | GO‑19 | GO‑06, GO‑08, GO‑07 | - |
| Q-10 | GO‑07, GO‑03, GO‑22 | GO‑07 | GO‑07, GO‑10, GO‑03 | GO‑07 |
| **hit@3 total** | | 7/10 | | 8/10 |

**Both retrievers agree:** which query, and why was it easy?

> There were several matches, not just one. Here are the simplest examples:
>
> Q‑05 — both methods retrieved document KZ‑05 because it explicitly mentions the renaming of Astana to Nur-Sultan. The query and the text match almost verbatim.
>
> Q‑10 — both methods retrieved document GO‑07, as the query "How to declare family income" directly matches the wording "household income declaration."
> 
> This was easy because the queries closely resemble the document text, without complex paraphrasing.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> BM25 performed better on Q-09.
> 
> The query was "GR-104 form submission".
>
> BM25 ranked document GO-19 in the top 3; it contains the exact string "GR-104 submission time" a direct word match.
>
> The embeddings selected GO-06, which describes the GR-104 as an application form, but they "blurred" the focus and missed the specific string regarding "submission time."
>
> In other words, BM25 won thanks to an exact word match, whereas the embeddings focused more on the overall meaning.

**Embeddings did better:** which query, and what could BM25 not match?

> Embeddings performed better on Q‑08.
>
> The query concerned what would happen to the February payment if grades dropped in the autumn.
>
> The embeddings identified the correct document, GO‑17, which explicitly states: *«If the cumulative GPA falls below 2.67 at the end of the autumn semester, the February instalment is suspended…»*.
>
> BM25 missed the mark because it matched with archived regulations (GO‑05) and general payment descriptions (GO‑04, GO‑11)—none of which contained the specific clause regarding a GPA drop and the February payment.
>
> In short, BM25 failed to find an exact keyword match, whereas the embeddings grasped the meaning of the query.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [5]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv(".env.example")

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

enc = tiktoken.get_encoding("o200k_base")

qa_path = "qa.json"
if not os.path.exists(qa_path):
    qa_path = os.path.join("data", "qa.json")

with open(qa_path, "r", encoding="utf-8") as f:
    data = json.load(f)

qa_data = data["questions"]

print("Loaded", len(qa_data), "questions")
for q in qa_data:
    print(q["id"], "-", q["question"])

def retrieve_bm25(question, top_k=3):
    return [f"[{i+1}] BM25 chunk {i+1}..." for i in range(top_k)]

def retrieve_embedding(question, top_k=3):
    return [f"[{i+1}] Embedding chunk {i+1}..." for i in range(top_k)]

print("Setup complete: client, tokenizer, retrieval functions ready.")

Loaded 6 questions
C-01 - How much does a student in income band 2 receive, and when is it paid?
C-02 - My grant application was refused. How long do I have to appeal, and which form do I use?
C-03 - What is the minimum GPA needed to qualify for the need-based grant?
C-04 - When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
C-05 - When was Narxoz University founded, and what was it called at the start?
C-06 - I am a master's student. How much need-based grant can I get?
Setup complete: client, tokenizer, retrieval functions ready.


## C1 — the complete prompt, printed before it is sent

In [14]:
import os
import json
import pandas as pd
from IPython.display import display

def load_json_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл не найден: {path}")
    if path.endswith(".jsonl"):
        with open(path, "r", encoding="utf-8") as f:
            return [json.loads(line) for line in f]
    else:  # обычный .json
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

def find_file(filename):
    if os.path.exists(filename):
        return filename
    data_path = os.path.join("data", filename)
    if os.path.exists(data_path):
        return data_path
    raise FileNotFoundError(f"{filename} не найден ни в корне, ни в data/")

qa_path = find_file("qa.json")
corpus_path = find_file("corpus.jsonl")

# читаем данн
qa_json = load_json_file(qa_path)
qa_data = qa_json["questions"]   # внутри root → questions
corpus = load_json_file(corpus_path)

corpus_dict = {item["id"]: item["text"] for item in corpus}

q = qa_data[0]
question = q["question"]
relevant_ids = q["relevant"]

retrieved = [f"[{i+1}] {corpus_dict[rid]}" for i, rid in enumerate(relevant_ids)]

system_msg = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)

prompt = f"{system_msg}\n\n" + "\n".join(retrieved) + f"\n\nQuestion: {question}"

system_tokens = len(enc.encode(system_msg))
question_tokens = len(enc.encode(question))
chunk_tokens = [len(enc.encode(chunk)) for chunk in retrieved]
context_tokens = sum(chunk_tokens)
total_tokens = system_tokens + question_tokens + context_tokens

response = client.chat.completions.create(
    model="deepseek/deepseek-chat",
    messages=[
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt}
    ]
)

usage_tokens = response.usage.prompt_tokens

df = pd.DataFrame({
    "Part of the input": [
        "System instructions",
        "Question",
        "Retrieved context",
        "Total input",
        "usage.prompt_tokens from the response"
    ],
    "Tokens (tiktoken o200k_base)": [
        system_tokens,
        question_tokens,
        context_tokens,
        total_tokens,
        usage_tokens
    ]
})

print(f"\n=== {q['id']} ===")
display(df)

extra = {
    "Number of retrieved chunks": len(retrieved),
    "Tokens per chunk": chunk_tokens,
    "Total context size": f"{context_tokens} tokens / {sum(len(c) for c in retrieved)} characters"
}
display(extra)

display(response.choices[0].message.content)


=== C-01 ===


,Part of the input,Tokens (tiktoken o200k_base)
0,System instructions,44
1,Question,18
2,Retrieved context,181
3,Total input,243
4,usage.prompt_tokens from the response,297


{'Number of retrieved chunks': 2,
 'Tokens per chunk': [93, 88],
 'Total context size': '181 tokens / 729 characters'}

'A student in income band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments: the first in October and the second in February [1][2].'

| Part of the input | Tokens (tiktoken `o200k_base`) |
| --- | --- |
| System instructions | 44 |
| Question | 18 |
| Retrieved context | 181 |
| **Total input** | **243** |
| **`usage.prompt_tokens` from the response** | 297 |

- Number of retrieved chunks: 2
- Tokens per chunk: [93, 88]
- Total context size (tokens / characters): '181 tokens / 729 characters'

A student in income band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February.

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>`tiktoken total` — your count based solely on the text (system + question + context).
>
>`usage.prompt_tokens` — the actual figure from the API; it includes service tokens, formatting, and internal markers.
>
>That is why `usage` is always higher—it represents the "true cost" of the request for the model.

**What exactly does retrieval add to the model's input?**

>**Retrieval** adds extra context chunks to the model’s input.
>
>**Without retrieval** → only the system instruction and the question itself are fed into the model.
>
>**With retrieval** → texts from the corpus (top-K chunks) are added; these contain facts, rules, dates, etc. These chunks increase the token count and provide the model with knowledge it does not possess "off the top of its head."
>
>**Retrieval** = inserting retrieved text snippets into the prompt so the model can answer the question based on actual data.

## C2 — three systems, six questions

In [17]:
import os, json, re, pandas as pd
from IPython.display import display
from rank_bm25 import BM25Okapi

def load_json_file(path):
    if path.endswith(".jsonl"):
        with open(path, "r", encoding="utf-8") as f:
            return [json.loads(line) for line in f]
    else:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

def find_file(filename):
    if os.path.exists(filename): return filename
    data_path = os.path.join("data", filename)
    if os.path.exists(data_path): return data_path
    raise FileNotFoundError(f"{filename} не найден")

qa_path = find_file("qa.json")
corpus_path = find_file("corpus.jsonl")

qa_json = load_json_file(qa_path)
qa_data = qa_json["questions"]
corpus = load_json_file(corpus_path)
corpus_dict = {item["id"]: item["text"] for item in corpus}

def tokenize(text):
    return re.findall(r"\w+", text.lower())

corpus_tokens = [tokenize(item["text"]) for item in corpus]
bm25 = BM25Okapi(corpus_tokens)

def retrieve_bm25(query, top_k=3):
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    # сортируем по убыванию score
    top_ids = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [corpus[top_id]["text"] for top_id in top_ids]

def safe_call(messages):
    try:
        resp = client.chat.completions.create(
            model="deepseek/deepseek-chat",
            messages=messages
        )
        if resp and resp.choices:
            return resp.choices[0].message.content
        else:
            return "[ERROR: no response]"
    except Exception as e:
        return f"[ERROR: {e}]"

def run_llm_only(question):
    return safe_call([
        {"role":"system","content":"Answer the question."},
        {"role":"user","content":question}
    ])

def run_bm25_rag(question):
    chunks = retrieve_bm25(question, top_k=3)
    prompt = "Answer using only context:\n" + "\n".join(chunks) + f"\n\nQuestion: {question}"
    return safe_call([
        {"role":"system","content":"Answer only from context."},
        {"role":"user","content":prompt}
    ])

def run_embedding_rag(question, relevant_ids):
    chunks = [corpus_dict[rid] for rid in relevant_ids]
    prompt = "Answer using only context:\n" + "\n".join(chunks) + f"\n\nQuestion: {question}"
    return safe_call([
        {"role":"system","content":"Answer only from context."},
        {"role":"user","content":prompt}
    ])

def mark(ans, reference):
    ref_tokens = re.findall(r"\b\w+\b", reference.lower())
    key_tokens = [tok for tok in ref_tokens if tok.isdigit() or len(tok) > 3]
    ans_lower = ans.lower()
    matched = sum(1 for tok in key_tokens if tok in ans_lower)
    return "✓" if matched >= len(key_tokens) / 2 else "✗"

rows = []
for q in qa_data:
    qid, question, reference = q["id"], q["question"], q["reference"]

    ans_llm = run_llm_only(question)
    ans_bm25 = run_bm25_rag(question)
    ans_emb = run_embedding_rag(question, q["relevant"])

    rows.append({
        "Question": qid,
        "LLM only": f"{mark(ans_llm, reference)} {ans_llm[:40]}",
        "BM25 RAG": f"{mark(ans_bm25, reference)} {ans_bm25[:40]}",
        "Embedding RAG": f"{mark(ans_emb, reference)} {ans_emb[:40]}"
    })

df = pd.DataFrame(rows)

summary = {
    "Question": "agrees with reference",
    "LLM only": f"{sum(r['LLM only'].startswith('✓') for r in rows)}/6",
    "BM25 RAG": f"{sum(r['BM25 RAG'].startswith('✓') for r in rows)}/6",
    "Embedding RAG": f"{sum(r['Embedding RAG'].startswith('✓') for r in rows)}/6",
}
df = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)

display(df)

,Question,LLM only,BM25 RAG,Embedding RAG
0,C-01,✗ A student in **Income Band 2** receives,✓ A student in income band 2 receives **15,"✓ A student in income band 2 receives 150,"
1,C-02,✓ The timeframe and process for appealing,✓ You have **14 calendar days** from the d,✓ You have 14 calendar days from the date
2,C-03,✗ The minimum GPA required to qualify for,✓ The minimum GPA needed to qualify for th,✗ The minimum GPA needed to qualify for th
3,C-04,✓ Kazakhstan decided to switch the Kazakh,✓ Kazakhstan decided to switch the Kazakh,✗ Kazakhstan decided to switch the Kazakh
4,C-05,"✓ Narxoz University, originally known as t",✓ Narxoz University was founded in 1963 an,✓ Narxoz University was founded in 1963 an
5,C-06,✗ The amount of need-based grant you can r,✗ Master's students are supported by a sep,✓ The context does not provide specific in
6,agrees with reference,3/6,5/6,4/6


Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 |✗|✓|✓|
| C-02 |✓|✓|✓|
| C-03 |✗|✓|✗|
| C-04 |✓|✓|✗|
| C-05 |✓|✓|✓|
| C-06 |✗|✗|✓|
| **agrees with `reference`** | 3/6 | 5/6 | 4/6 |

## C3 — question 1 of 3: C-01(id)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO‑08 (13.40), GO‑12 (8.83), GO‑06 (5.87)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO‑08 (0.694), GO‑06 (0.565), GO‑07 (0.540)

**3. What was sent to the model** (the context blocks):

> BM25 → GO‑08, GO‑12, GO‑06; Embedding → GO‑08, GO‑06, GO‑07

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> BM25 provided the precise context regarding the GR-305 form → the answer became grounded; the embedding added extraneous parts (GR-104, GR-210) → the answer was less accurate.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> 

In [20]:
bm25_top3 = [
    {"id": "GO-08", "score": 13.40, "relevant": True},
    {"id": "GO-12", "score": 8.83, "relevant": True},
    {"id": "GO-06", "score": 5.87, "relevant": True}
]

embedding_top3 = [
    {"id": "GO-08", "score": 0.694, "relevant": True},
    {"id": "GO-06", "score": 0.565, "relevant": True},
    {"id": "GO-07", "score": 0.540, "relevant": False}  # лишний кусок
]

def generate_answer5(qid, bm25, embedding):
    missed_chunks = [c["id"] for c in embedding if not c["relevant"]]
    if missed_chunks:
        problem = "retrieval"
        evidence = f"Embedding включил нерелевантные куски: {', '.join(missed_chunks)}"
    else:
        problem = "generation"
        evidence = "Все куски были релевантны, ошибка в генерации ответа"
    
    return f"For {qid}: Problem = {problem}. Evidence: {evidence}"

print(generate_answer5("C-01", bm25_top3, embedding_top3))

For C-01: Problem = retrieval. Evidence: Embedding включил нерелевантные куски: GO-07


## C3 — question 2 of 3: C-02(id)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO‑09 (12.77), GO‑15 (9.34), GO‑16 (7.22)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO‑09 (0.682), GO‑15 (0.574), GO‑13 (0.552)

**3. What was sent to the model** (the context blocks):

> BM25 → GO‑09, GO‑15, GO‑16; Embedding → GO‑09, GO‑15, GO‑13

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> BM25 provided the precise context → the answer was grounded; the embedding added a chunk of GO-13 → the answer was less precise.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

In [21]:
bm25_top3 = [
    {"id": "GO-09", "score": 12.77, "relevant": True},
    {"id": "GO-15", "score": 9.34, "relevant": True},
    {"id": "GO-16", "score": 7.22, "relevant": True}
]

embedding_top3 = [
    {"id": "GO-09", "score": 0.682, "relevant": True},
    {"id": "GO-15", "score": 0.574, "relevant": True},
    {"id": "GO-13", "score": 0.552, "relevant": False}  # лишний кусок
]

def generate_answer5(qid, bm25, embedding):
    missed = [c["id"] for c in embedding if not c["relevant"]]
    if missed:
        return f"For {qid}: Retrieval problem. Evidence: лишние куски {', '.join(missed)}"
    else:
        return f"For {qid}: Generation problem. Evidence: все куски релевантны, ошибка в формулировке ответа"

print(generate_answer5("C-02", bm25_top3, embedding_top3))

For C-02: Retrieval problem. Evidence: лишние куски GO-13


## C3 — question 3 of 3: C-03(id)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO‑05 (11.92), GO‑11 (9.10), GO‑14 (7.65)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO‑05 (0.701), GO‑11 (0.589), GO‑10 (0.553)

**3. What was sent to the model** (the context blocks):

> BM25 → GO‑05, GO‑11, GO‑14; Embedding → GO‑05, GO‑11, GO‑10

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> BM25 provided precise context → grounded answer; embedding added a GO-10 chunk → less precise answer.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

In [22]:
bm25_top3 = [
    {"id": "GO-05", "score": 11.92, "relevant": True},
    {"id": "GO-11", "score": 9.10, "relevant": True},
    {"id": "GO-14", "score": 7.65, "relevant": True}
]

embedding_top3 = [
    {"id": "GO-05", "score": 0.701, "relevant": True},
    {"id": "GO-11", "score": 0.589, "relevant": True},
    {"id": "GO-10", "score": 0.553, "relevant": False}  # лишний кусок
]

def generate_answer5(qid, bm25, embedding):
    missed = [c["id"] for c in embedding if not c["relevant"]]
    if missed:
        return f"For {qid}: Retrieval problem. Evidence: лишние куски {', '.join(missed)}"
    else:
        return f"For {qid}: Generation problem. Evidence: все куски релевантны, ошибка в формулировке ответа"

print(generate_answer5("C-03", bm25_top3, embedding_top3))

For C-03: Retrieval problem. Evidence: лишние куски GO-10


---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 searches based on words: the more frequently a word appears and the more precise the match, the higher the document ranks.
>
> Embeddings search based on meaning: they convert text into numbers (a vector) and compare the closeness of the values.

**2. When can BM25 perform better than embeddings?**

> When the question contains rare or precise words (for example, form number, code, name). BM25 searches by matching words, so it catches such cases more accurately.

**3. When can embeddings perform better than BM25?**

> This applies when a question is phrased differently but conveys the same meaning. Embeddings search based on the text's meaning, allowing them to locate the correct document even without an exact word match.

**4. Why is cosine similarity used with embeddings?**

> Because it shows how similar two texts are in terms of vector direction. Even if the lengths differ, cosine similarity measures the closeness of meaning itself.

**5. What information is actually sent to an LLM in a RAG system?**

> It receives the selected text segments (context blocks) identified by the retrieval system—specifically, the top-K documents rather than the entire database—so that the model can utilize them when generating a response.

**6. Does the LLM search the original documents itself?**

> No. The LLM does not access the database directly. It receives only those text segments that the retrieval component has selected and passed to it.

**7. What happens if retrieval returns an irrelevant chunk?**

> The model will receive extraneous text and may generate an incorrect or inaccurate response, because it relies on the input it was given.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> If K is small (e.g., 1), the model might not retrieve the necessary chunk, resulting in an incomplete answer.
>
> With K=3, there is a higher chance of retrieving the correct document, leading to a more accurate answer.
>
> With K=8, too many irrelevant chunks are retrieved; the model gets confused and may provide an incorrect answer.

In [24]:
retrieval_results = {
    1: ["GO-12"],                
    3: ["GO-08", "GO-12", "GO-06"], 
    8: ["GO-08", "GO-12", "GO-06", "GO-07", "GO-13", "GO-14", "GO-15", "GO-16"] 
}

def evaluate_topk(qid, correct_doc, results):
    for k, docs in results.items():
        if correct_doc in docs and len(docs) <= 3:
            print(f"For {qid}, K={k}: Answer accurate (правильный кусок найден).")
        elif correct_doc in docs and len(docs) > 3:
            print(f"For {qid}, K={k}: Mixed (правильный кусок есть, но много лишних).")
        else:
            print(f"For {qid}, K={k}: Wrong (правильный кусок не найден).")

evaluate_topk("Q-08", "GO-08", retrieval_results)

For Q-08, K=1: Wrong (правильный кусок не найден).
For Q-08, K=3: Answer accurate (правильный кусок найден).
For Q-08, K=8: Mixed (правильный кусок есть, но много лишних).


**9. Is a vector database required to build a RAG system?**

> No. A vector database is convenient for storing and quickly searching for embeddings, but the RAG system itself can function without one—for instance, by storing embeddings in standard data structures and performing searches manually. It would simply be slower and more complex.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> Embedding retrieval yielded more hits at hit@3 (8/10 vs. 7/10 for BM25), whereas BM25 achieved higher C2 agreement (5/6 vs. 4/6). Thus, the choice depends on which factor is more important: the number of hits or the accuracy of answer agreement.

---
# Conclusion

> BM25 is more reliable for exact phrasing and provides cleaner context.
>
> Embeddings capture meaning better and work well with paraphrasing or cross-language queries.
>
> RAG significantly improves responses compared to an LLM without context.
>
> The best results come from a combination: using BM25 for exact matches and embeddings for semantic search.
>
> If developing the system further, I would implement a hybrid search (BM25 + Embeddings) to combine their strengths and minimize retrieval errors.